In [ ]:
import os
import random
import re

import numpy as np
import cv2

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.metrics import F1Score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# **[1] Load Data**

In [ ]:
DATA_DIR = os.path.join("data", "images")
IMAGE_SIZE = (224, 224)


def load_images(split_dir):
    features, labels, image_paths = [], [], []
    for member_name in sorted(os.listdir(split_dir)):
        member_dir = os.path.join(split_dir, member_name)
        if not os.path.isdir(member_dir):
            continue
        for file_name in os.listdir(member_dir):
            image_path = os.path.join(member_dir, file_name)
            image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue
            image = cv2.resize(image, IMAGE_SIZE)
            features.append(image.flatten())
            labels.append(member_name)
            image_paths.append(image_path)
    features = np.array(features, dtype=np.float32) / 255.0
    labels = np.array(labels)
    image_paths = np.array(image_paths)
    return features, labels, image_paths


X_train, y_train_label, paths_train = load_images(os.path.join(DATA_DIR, "train"))
X_test, y_test_label, paths_test = load_images(os.path.join(DATA_DIR, "test"))

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}, Fitur per data: {X_train.shape[1]}")
print(f"Kelas: {sorted(set(y_train_label))}")

#### Train/Val Split (Temporal, per anggota)

In [ ]:
VAL_RATIO = 0.15
VAL_GAP = 5  # frame dibuang di titik potong biar train/val gak near-duplicate


def _frame_index(path):
    match = re.search(r"_(\d+)\.\w+$", os.path.basename(path))
    return int(match.group(1)) if match else -1


train_idx, val_idx = [], []
for member_name in sorted(set(y_train_label)):
    member_indices = [i for i, label in enumerate(y_train_label) if label == member_name]
    member_indices.sort(key=lambda i: _frame_index(paths_train[i]))

    split_idx = int(len(member_indices) * (1 - VAL_RATIO))
    train_idx.extend(member_indices[:max(split_idx - VAL_GAP, 0)])
    val_idx.extend(member_indices[split_idx + VAL_GAP:])

train_idx = np.array(sorted(train_idx))
val_idx = np.array(sorted(val_idx))

X_val, y_val_label, paths_val = X_train[val_idx], y_train_label[val_idx], paths_train[val_idx]
X_train, y_train_label, paths_train = X_train[train_idx], y_train_label[train_idx], paths_train[train_idx]

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}")

# **[2] Preprocessing**

#### Augmentasi Data (Train Only)

In [ ]:
def augment_image(image):
    variants = []

    # flip horizontal
    variants.append(cv2.flip(image, 1))

    # rotasi kecil (15 derajat)
    height, width = image.shape
    for angle in (-15, 15):
        rotation_matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
        variants.append(cv2.warpAffine(image, rotation_matrix, (width, height), borderMode=cv2.BORDER_REPLICATE))

    # brightness
    variants.append(cv2.convertScaleAbs(image, alpha=1.2, beta=15))
    variants.append(cv2.convertScaleAbs(image, alpha=0.8, beta=-15))

    # translasi kecil
    shift_x, shift_y = int(0.05 * width), int(0.05 * height)
    translation_matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    variants.append(cv2.warpAffine(image, translation_matrix, (width, height), borderMode=cv2.BORDER_REPLICATE))

    return variants


def augment_train_set(image_paths, labels):
    augmented_features, augmented_labels, augmented_paths = [], [], []

    for image_path, label in zip(image_paths, labels):
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, IMAGE_SIZE)

        for variant in augment_image(image):
            augmented_features.append(variant.flatten())
            augmented_labels.append(label)
            augmented_paths.append(image_path + " (aug)")

    return (
        np.array(augmented_features, dtype=np.float32) / 255.0,
        np.array(augmented_labels),
        np.array(augmented_paths),
    )


X_train_aug, y_train_label_aug, paths_train_aug = augment_train_set(paths_train, y_train_label)

X_train = np.concatenate([X_train, X_train_aug], axis=0)
y_train_label = np.concatenate([y_train_label, y_train_label_aug], axis=0)
paths_train = np.concatenate([paths_train, paths_train_aug], axis=0)

print(f"Train setelah augmentasi: {X_train.shape[0]} data")

#### Encoding

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(y_train_label)

y_train = to_categorical(label_encoder.transform(y_train_label))
y_val = to_categorical(label_encoder.transform(y_val_label))
y_test = to_categorical(label_encoder.transform(y_test_label))

num_classes = y_train.shape[1]
num_features = X_train.shape[1]

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(label_encoder.transform(y_train_label)),
    y=label_encoder.transform(y_train_label)
)
class_weight_dict = dict(enumerate(class_weight_values))

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")
print(f"Class weight: {class_weight_dict}")

# **[3] Build Model**

In [ ]:
model = Sequential([
    Dense(128, activation="relu", input_shape=(num_features,)),
    Dropout(0.4),
    Dense(64, activation="relu"),
    Dropout(0.4),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=[F1Score(average="macro", name="f1_score")]
)

model.summary()

### Training

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    class_weight=class_weight_dict,
    verbose=1
)

# **[4] Evaluation**

#### Learning Curve (F1 Score & Loss)

In [ ]:
fig, (f1_ax, loss_ax) = plt.subplots(1, 2, figsize=(12, 4.5))

epochs_range = range(1, len(history.history["loss"]) + 1)

f1_ax.plot(epochs_range, history.history["f1_score"], label="Train F1", color="navy")
f1_ax.plot(epochs_range, history.history["val_f1_score"], label="Val F1", color="crimson")
f1_ax.set_title("F1 Score per Epoch")
f1_ax.set_xlabel("Epoch")
f1_ax.set_ylabel("F1 Score (macro)")
f1_ax.legend()
f1_ax.grid(alpha=0.3)

loss_ax.plot(epochs_range, history.history["loss"], label="Train Loss", color="navy")
loss_ax.plot(epochs_range, history.history["val_loss"], label="Val Loss", color="crimson")
loss_ax.set_title("Loss per Epoch")
loss_ax.set_xlabel("Epoch")
loss_ax.set_ylabel("Loss")
loss_ax.legend()
loss_ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

#### Evaluasi

In [ ]:
y_test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
y_test_true = np.argmax(y_test, axis=1)

print(classification_report(y_test_true, y_test_pred, target_names=label_encoder.classes_))

confusion_mat = confusion_matrix(y_test_true, y_test_pred)
confusion_mat_display = ConfusionMatrixDisplay(confusion_matrix=confusion_mat, display_labels=label_encoder.classes_)
confusion_mat_display.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# **[5] Inference**

In [ ]:
def predict_image(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, IMAGE_SIZE)
    image_input = image.flatten().astype(np.float32) / 255.0
    image_input = np.expand_dims(image_input, axis=0)

    prediction = model.predict(image_input, verbose=0)
    label = label_encoder.inverse_transform([np.argmax(prediction)])[0]
    confidence = np.max(prediction)
    return label, confidence, image

n_samples = 10
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(paths_test), size=n_samples, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 7))

for ax, idx in zip(axes.flatten(), sample_indices):
    image_path = paths_test[idx]
    true_label = label_encoder.inverse_transform([np.argmax(y_test[idx])])[0]

    pred_label, confidence, image = predict_image(image_path)
    is_correct = pred_label == true_label
    color = "green" if is_correct else "red"

    ax.imshow(image, cmap="gray")
    ax.axis("off")
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})",
                 color=color, fontsize=10)

plt.tight_layout()
plt.show()